In [ ]:
!pip install -q transformers datasets peft accelerate sentencepiece rouge-score nltk

  Preparing metadata (setup.py) ... done


In [ ]:
import torch
import torch.nn as nn
import numpy as np

from datasets import load_dataset
from torch.utils.data import Dataset

from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Trainer,
    TrainingArguments
)

from peft import LoraConfig, get_peft_model

from rouge_score import rouge_scorer

In [ ]:
# Setting the device here
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
dataset = load_dataset("raeidsaqur/NIFTY")

train_data = dataset["train"]
val_data   = dataset["valid"]
test_data  = dataset["test"]

tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [ ]:
class NIFTYDCCSDataset(Dataset):

    def __init__(self, data, tokenizer, max_len=512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        sample = self.data[idx]

        prompt = (
            f"news: {sample['news']} "
            f"context: {sample['context']} "
            f"predict market movement:"
        )

        # Combine label and pct_change as the target string
        target = (
            sample["conversations"][1]["value"]
            if "conversations" in sample
            else "Market summary unavailable."
        )

        inputs = self.tokenizer(
            prompt,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        labels = self.tokenizer(
            target,
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        label_ids = labels.input_ids.squeeze()
        label_ids[label_ids == self.tokenizer.pad_token_id] = -100   # Done for masking the Pad token.
        return {
            "input_ids": inputs.input_ids.squeeze(),
            "attention_mask": inputs.attention_mask.squeeze(),
            "labels": label_ids
        }

In [ ]:
class DCCST5(T5ForConditionalGeneration):

    def __init__(self, config):
        super().__init__(config)
        hidden = config.d_model
        self.last_alpha = None
        self.causal_gate = nn.Linear(hidden * 2, 1)

        # Initialize gate to output ~0.5 (neutral) at start
        nn.init.zeros_(self.causal_gate.weight)
        nn.init.zeros_(self.causal_gate.bias)

    @classmethod
    def _can_set_experts_implementation(cls):
        return False

    def forward(self, input_ids=None, attention_mask=None, labels=None, encoder_outputs=None, **kwargs):

      if encoder_outputs is None:
        encoder_outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )

        h = encoder_outputs.last_hidden_state
        B, T, H = h.size()
        mid = T // 2

        h_news   = h[:, :mid, :]
        h_market = h[:, mid:, :]

        alpha = torch.sigmoid(
            self.causal_gate(torch.cat([h_news.mean(dim=1), h_market.mean(dim=1)], dim=-1))
        ).view(B, 1, 1)

        self.last_alpha = alpha.detach().cpu()

        encoder_outputs.last_hidden_state = torch.cat(
            [alpha * h_news, (1 - alpha) * h_market], dim=1
        )

      return super().forward(
          encoder_outputs=encoder_outputs,
          attention_mask=attention_mask,
          labels=labels
      )
    def prepare_inputs_for_generation(
      self,
      input_ids,
      past_key_values=None,
      attention_mask=None,
      encoder_outputs=None,
      **kwargs
  ):
      return {
          "decoder_input_ids": input_ids,
          "past_key_values": past_key_values,
          "attention_mask": attention_mask,
          "encoder_outputs": encoder_outputs,
      }

In [ ]:
model = DCCST5.from_pretrained("t5-small")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q", "v"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

model = get_peft_model(model, lora_config)
model = model.to(device)
model.print_trainable_parameters()

In [ ]:
train_dataset = NIFTYDCCSDataset(train_data, tokenizer)
val_dataset   = NIFTYDCCSDataset(val_data, tokenizer)

In [ ]:
training_args = TrainingArguments(
    output_dir="./dccs_nifty_t5",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=20,
    logging_steps=50,
    eval_strategy="steps",
    remove_unused_columns=False,
    eval_steps=200,
    save_steps=200,
    fp16=False,

    # fp16_full_eval=True,
    report_to="none"
)

In [ ]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    processing_class=tokenizer
)

In [ ]:
print(dataset["train"].column_names)
print(dataset["train"][0])

In [ ]:
# Manual forward pass check
batch = next(iter(torch.utils.data.DataLoader(train_dataset, batch_size=2)))
model.train()
out = model(
    input_ids=batch["input_ids"].to(model.device),
    attention_mask=batch["attention_mask"].to(model.device),
    labels=batch["labels"].to(model.device)
)
print("Loss:", out.loss)
print("Loss dtype:", out.loss.dtype)

In [ ]:
trainer.train()
import pandas as pd
# Convert logs to DataFrame for easy analysis
logs = pd.DataFrame(trainer.state.log_history)
print(logs[['step', 'loss', 'eval_loss']].dropna())


In [ ]:

trainer.save_model("./dccs_nifty_t5_final")

tokenizer.save_pretrained("./dccs_nifty_t5_final")


In [ ]:
model.eval()

sample = test_data[0]

prompt = (
    f"news: {sample['news']} "
    f"context: {sample['context']} "
    f"summarize market movement:"
)

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
).to(device)

with torch.no_grad():

    outputs = model.generate(
        **inputs,
        max_length=120
    )

prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(prediction)
print("α-gate value:", model.last_alpha.item())

In [ ]:
scorer = rouge_scorer.RougeScorer(["rouge1","rougeL"], use_stemmer=True)

rouge1_scores = []
rougeL_scores = []

for i in range(50):

    sample = test_data[i]

    prompt = (
        f"news: {sample['news']} "
        f"context: {sample['context']} "
        f"summarize market movement:"
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=120)

    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)

    scores = scorer.score(sample["summary"], pred)

    rouge1_scores.append(scores["rouge1"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

print("ROUGE1:", np.mean(rouge1_scores))
print("ROUGEL:", np.mean(rougeL_scores))